In [3]:
import os
import re
import json
import shutil
import random
import warnings
import numpy as np
import pandas as pd
import unicodedata

import os
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from collections import Counter
from sklearn.metrics import accuracy_score, f1_score, log_loss, classification_report


import nltk
from nltk.corpus import stopwords

from collections import Counter
from joblib import dump, load

from sklearn.pipeline import Pipeline
from sklearn.calibration import CalibratedClassifierCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer as SklearnTfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    log_loss,
    confusion_matrix
)
from sklearn.model_selection import StratifiedKFold, cross_validate


import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding,
    EarlyStoppingCallback
)

from datasets import Dataset





warnings.filterwarnings("ignore")

nltk.download("stopwords")
STOPWORD = set(stopwords.words("french"))

RANDOM_STATE = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [17]:
class TransformerWrapper:
    def __init__(
        self,
        model_name="distilbert-base-uncased", # "uncased" est souvent plus stable que multilingual
        max_len=256,   # Augmenté à 256 pour de meilleures performances sur IMDb
        batch_size=16,
        lr=2e-5,
        epochs=3,
        output_dir="./results_distilbert"
    ):
        self.model_name = model_name
        self.max_len = max_len
        self.batch_size = batch_size
        self.lr = lr
        self.epochs = epochs
        self.output_dir = output_dir

        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(
            model_name,
            num_labels=2
        )

        self.trainer = None

    def _tokenize_function(self, examples):
        return self.tokenizer(
            examples["text"],
            truncation=True,
            max_length=self.max_len
        )

    def fit(self, texts, y, texts_valid=None, y_valid=None):
        # Préparation des datasets
        train_ds = HFDataset.from_dict({
            "text": list(texts),
            "labels": list(map(int, y))
        }).map(self._tokenize_function, batched=True, remove_columns=["text"])

        eval_ds = None
        if texts_valid is not None and y_valid is not None:
            eval_ds = HFDataset.from_dict({
                "text": list(texts_valid),
                "labels": list(map(int, y_valid))
            }).map(self._tokenize_function, batched=True, remove_columns=["text"])

        data_collator = DataCollatorWithPadding(tokenizer=self.tokenizer)

        # Calcul du warmup
        steps_per_epoch = max(1, len(train_ds) // self.batch_size)
        warmup_steps = int(steps_per_epoch * self.epochs * 0.1)

        args = TrainingArguments(
            output_dir=self.output_dir,
            num_train_epochs=self.epochs,
            learning_rate=self.lr,
            per_device_train_batch_size=self.batch_size,
            per_device_eval_batch_size=self.batch_size,
            weight_decay=0.01,
            warmup_steps=warmup_steps,
            logging_strategy="epoch",
            eval_strategy="epoch" if eval_ds is not None else "no",
            save_strategy="epoch" if eval_ds is not None else "no",
            load_best_model_at_end=True,
            metric_for_best_model="f1",
            fp16=True, # DistilBERT gère très bien le fp16
            report_to="none",
            seed=RANDOM_STATE
        )

        self.trainer = Trainer(
            model=self.model,
            args=args,
            train_dataset=train_ds,
            eval_dataset=eval_ds,
            processing_class=self.tokenizer,
            data_collator=data_collator,
            compute_metrics=hf_compute_metrics,
            callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
        )

        self.trainer.train()
        self.trainer.save_model(self.output_dir)
        return self

    def predict_proba(self, texts):
        ds = HFDataset.from_dict({"text": list(texts)}).map(
            self._tokenize_function, batched=True, remove_columns=["text"]
        )
        preds = self.trainer.predict(ds)
        return torch.softmax(torch.tensor(preds.predictions), dim=1).numpy()

    def predict(self, texts):
        return np.argmax(self.predict_proba(texts), axis=1)

def hf_compute_metrics(eval_pred):
  logits, labels = eval_pred
  probs = torch.softmax(torch.tensor(logits), dim=1).numpy()
  preds = np.argmax(probs, axis=1)

  return {
      "accuracy": accuracy_score(labels, preds),
      "f1": f1_score(labels, preds),
      "precision": precision_score(labels, preds, zero_division=0),
      "recall": recall_score(labels, preds, zero_division=0),
  }

In [7]:
file = '/content/json_pol.json'
with open(file,encoding="utf-8") as f:
  json_data = json.load(f)

json_texts = [x[0] for x in json_data]
json_labels = [x[1] for x in json_data]


def clean_simple(txt):
    txt = re.sub(r"<[^>]+>", " ", txt)
    txt = unicodedata.normalize('NFC', txt)

    return re.sub(r"\s+", " ", txt).strip()

json_texts = [clean_simple(t) for t in json_texts]



In [8]:


X_train, X_val, y_train, y_val = train_test_split(
    json_texts, json_labels, test_size=0.2, random_state=42, stratify=json_labels, shuffle=True
)


train_dis_dataset = Dataset.from_dict({"text": X_train, "label": y_train})
test_dis_dataset = Dataset.from_dict({"text": X_val, "label": y_val})


In [13]:
trainer = TransformerWrapper(
    model_name="distilbert-base-uncased",
    max_len=512,
    batch_size=16,
    lr=1e-5,
    epochs=3
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [18]:
trainer.fit(
    texts=train_dis_dataset["text"],
    y=train_dis_dataset["label"],
    texts_valid=test_dis_dataset["text"],
    y_valid=test_dis_dataset["label"]
)

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Map:   0%|          | 0/5000 [00:00<?, ? examples/s]

Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.316461,0.211707,0.919200,0.920629,0.904633,0.937200
2,0.166387,0.239346,0.926400,0.927928,0.909056,0.947600
3,0.119495,0.283237,0.925000,0.925075,0.924152,0.926000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [19]:
def load_test_file(test_file):
    texts = []
    with open(test_file, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            line = line.strip()
            if line:
                texts.append(line)
    return texts

final_test_texts = load_test_file('/content/testSentiment.txt')


In [20]:
final_preds = trainer.predict(final_test_texts)

mapped_preds = ["P" if p == 1 else "N" for p in final_preds]

df = pd.DataFrame(mapped_preds)
df.to_csv("predictions.csv", index=False, header=False)

Parameter 'function'=<bound method TransformerWrapper._tokenize_function of <__main__.TransformerWrapper object at 0x7c84f6876540>> of the transform datasets.arrow_dataset.Dataset._map_single couldn't be hashed properly, a random hash was used instead. Make sure your transforms and parameters are serializable with pickle or dill for the dataset fingerprinting and caching to work. If you reuse this transform, the caching mechanism will consider it to be different from the previous calls and recompute everything. This warning is only showed once. Subsequent hashing failures won't be showed.


Map:   0%|          | 0/25000 [00:00<?, ? examples/s]